# Laboratorium 8: Transformata Hit-or-miss oraz Kontury obiektów

**Politechnika Świętokrzyska**  
**Centrum Laserowych Technologii Metali**  
**Automatyka i Robotyka - 2 semestr II stopnia**

---

## Spis treści
1. [Teoria](#teoria)
2. [Kod startowy](#kod-startowy)
3. [Zadania do wykonania](#zadania)
4. [Przykłady użycia](#przyklady)
5. [Podsumowanie](#podsumowanie)


## 1. Teoria <a id='teoria'></a>

### 1.1 Wprowadzenie

W poprzednich ćwiczeniach poznaliśmy podstawowe operacje morfologiczne: erozję, dylatację, otwarcie i zamknięcie. Dzisiaj skupimy się na bardziej zaawansowanych operacjach:

- **Transformata Hit-or-Miss** - wykrywanie specyficznych wzorców w obrazie
- **Ekstrakcja konturów** - wydobywanie krawędzi obiektów
- **Wypełnianie dziur** - uzupełnianie pustych obszarów wewnątrz obiektów
- **Wykrywanie narożników** - identyfikacja punktów charakterystycznych

**Zastosowania:**
- Rozpoznawanie wzorców
- Analiza kształtu obiektów
- Segmentacja obrazu
- Ekstrakcja cech geometrycznych
- Inspekcja wizualna w przemyśle


### 1.2 Transformata Hit-or-Miss

Transformata Hit-or-Miss (HMT) służy do wykrywania specyficznych wzorców w obrazie binarnym. W przeciwieństwie do zwykłej erozji, która sprawdza tylko obecność obiektu, HMT sprawdza **jednocześnie** obecność obiektu (pierwszego planu) i tła.

#### Definicja matematyczna:

$$A \circledast B = (A \ominus B_1) \cap (A^c \ominus B_2)$$

gdzie:
- $A$ - obraz binarny
- $B = (B_1, B_2)$ - para elementów strukturalnych
- $B_1$ - element "hit" (wzorzec dla pierwszego planu)
- $B_2$ - element "miss" (wzorzec dla tła)
- $A^c$ - dopełnienie obrazu $A$ (negacja)
- $\ominus$ - erozja
- $\cap$ - iloczyn logiczny (AND)

#### Interpretacja:

Piksel wyjściowy = 1 wtedy i tylko wtedy, gdy:
1. Element $B_1$ **pasuje** do obiektu (pierwszego planu)
2. Element $B_2$ **pasuje** do tła (dopełnienia obiektu)

#### Element strukturalny 2-wartościowy:

W praktyce używamy **pary elementów strukturalnych** z **dwoma wartościami** (0 i 1):
- **SE1 (hit)**: `1` - musi być obiekt (pierwszy plan), `0` - nie ma znaczenia
- **SE2 (miss)**: `1` - musi być tło, `0` - nie ma znaczenia

**Przykład - wykrywanie lewego górnego narożnika:**

```
SE1 (hit):          SE2 (miss):
┌────────┐          ┌────────┐
│ 0  0  0 │          │ 1  1  1 │
│ 0  1  1 │          │ 1  0  0 │
│ 0  1  0 │          │ 1  0  0 │
└────────┘          └────────┘
```

#### Algorytm:

```
1. Przygotuj B1 (piksele = 1 w SE hit) i B2 (piksele = 1 w SE miss)
2. Wykonaj erozję: E1 = A ⊖ B1
3. Wykonaj erozję dopełnienia: E2 = A^c ⊖ B2
4. Wynik: HMT = E1 ∩ E2
```


### 1.3 Ekstrakcja konturów

Kontury (krawędzie) obiektów można wydobyć za pomocą operacji morfologicznych.

#### Kontur zewnętrzny (External Boundary):

$$\beta_e(A) = A - (A \ominus B)$$

Wydobywa **zewnętrzną** krawędź obiektu (piksele na brzegu od strony obiektu).

#### Kontur wewnętrzny (Internal Boundary):

$$\beta_i(A) = (A \oplus B) - A$$

Wydobywa **wewnętrzną** krawędź obiektu (piksele na brzegu od strony tła).

#### Gradient morfologiczny (Morphological Gradient):

$$\beta(A) = (A \oplus B) - (A \ominus B)$$

Wydobywa **pełny** kontur obiektu (grubszy niż poprzednie metody).


### 1.4 Wypełnianie dziur (Hole Filling)

Dziura to obszar tła otoczony przez obiekt. Wypełnianie dziur jest przydatne w:
- Poprawie segmentacji
- Analizie kształtu
- Przygotowaniu obrazu do dalszego przetwarzania

#### Algorytm iteracyjny:

$$X_k = (X_{k-1} \oplus B) \cap A^c$$

gdzie:
- $X_0$ - punkt startowy wewnątrz dziury
- $B$ - element strukturalny (zwykle krzyż 3×3)
- $A^c$ - dopełnienie obrazu (tło)
- Iteracja do momentu: $X_k = X_{k-1}$

**Wynik końcowy:** $H = A \cup X_k$


### 1.5 Wykrywanie narożników

Narożniki to punkty charakterystyczne obiektów, przydatne w:
- Rozpoznawaniu kształtów
- Dopasowywaniu wzorców
- Analizie geometrycznej

#### Typy narożników:

Dla prostokątów wykrywamy 4 typy narożników używając **par elementów strukturalnych**:

**1. Lewy górny:**
```
SE1 (hit):       SE2 (miss):
│ 0  0  0 │      │ 1  1  1 │
│ 0  1  1 │      │ 1  0  0 │
│ 0  1  0 │      │ 1  0  0 │
```

**2. Prawy górny:**
```
SE1 (hit):       SE2 (miss):
│ 0  0  0 │      │ 1  1  1 │
│ 1  1  0 │      │ 0  0  1 │
│ 0  1  0 │      │ 0  0  1 │
```

**3. Lewy dolny:**
```
SE1 (hit):       SE2 (miss):
│ 0  1  0 │      │ 1  0  0 │
│ 0  1  1 │      │ 1  0  0 │
│ 0  0  0 │      │ 1  1  1 │
```

**4. Prawy dolny:**
```
SE1 (hit):       SE2 (miss):
│ 0  1  0 │      │ 0  0  1 │
│ 1  1  0 │      │ 0  0  1 │
│ 0  0  0 │      │ 1  1  1 │
```


## 2. Kod startowy <a id='kod-startowy'></a>

### 2.1 Import bibliotek


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import cv2

plt.rcParams['figure.figsize'] = (15, 10)
plt.rcParams['image.cmap'] = 'gray'

print("✓ Biblioteki załadowane")

### 2.2 Funkcje pomocnicze


In [ ]:
def show_images(images, titles=None, figsize=(15, 5), cmap='gray'):
    n = len(images)
    fig, axes = plt.subplots(1, n, figsize=figsize)
    if n == 1:
        axes = [axes]
    for i, (img, ax) in enumerate(zip(images, axes)):
        ax.imshow(img, cmap=cmap)
        ax.axis('off')
        if titles and i < len(titles):
            ax.set_title(titles[i], fontsize=12, fontweight='bold')
    plt.tight_layout()
    plt.show()

def show_images_grid(images, titles=None, rows=2, cols=4, figsize=(15, 8), cmap='gray'):
    fig, axes = plt.subplots(rows, cols, figsize=figsize)
    axes = axes.flatten()
    for i, ax in enumerate(axes):
        if i < len(images):
            ax.imshow(images[i], cmap=cmap)
            if titles and i < len(titles):
                ax.set_title(titles[i], fontsize=10, fontweight='bold')
        ax.axis('off')
    plt.tight_layout()
    plt.show()

print("✓ Funkcje pomocnicze zdefiniowane")

### 2.3 Elementy strukturalne


In [ ]:
kernel_cross_3 = cv2.getStructuringElement(cv2.MORPH_CROSS, (3, 3))
kernel_rect_3 = cv2.getStructuringElement(cv2.MORPH_RECT, (3, 3))
kernel_rect_5 = cv2.getStructuringElement(cv2.MORPH_RECT, (5, 5))

print("Elementy strukturalne:")
print("\nKrzyż 3×3:")
print(kernel_cross_3)
print("\nKwadrat 3×3:")
print(kernel_rect_3)

### 2.4 Wklejenie funkcji z poprzednich zajęć

**UWAGA:** Wklej tutaj swoje implementacje funkcji z poprzednich laboratoriów:
- `erode(image, kernel)` - erozja
- `dilate(image, kernel)` - dylatacja
- `opening(image, kernel)` - otwarcie
- `closing(image, kernel)` - zamknięcie


In [ ]:
# ============================================================================
# WKLEJ TUTAJ SWOJE FUNKCJE Z POPRZEDNICH ZAJĘĆ
# ============================================================================

def erode(image, kernel):
    """
    Erozja morfologiczna.
    Wklej swoją implementację z poprzednich zajęć.
    """
    # TODO: Wklej swoją implementację
    pass

def dilate(image, kernel):
    """
    Dylatacja morfologiczna.
    Wklej swoją implementację z poprzednich zajęć.
    """
    # TODO: Wklej swoją implementację
    pass

def opening(image, kernel):
    """
    Otwarcie morfologiczne.
    Wklej swoją implementację z poprzednich zajęć.
    """
    # TODO: Wklej swoją implementację
    pass

def closing(image, kernel):
    """
    Zamknięcie morfologiczne.
    Wklej swoją implementację z poprzednich zajęć.
    """
    # TODO: Wklej swoją implementację
    pass

# ============================================================================


### 2.5 Obrazy testowe


In [ ]:
img_rectangle = np.zeros((100, 100), dtype=np.uint8)
cv2.rectangle(img_rectangle, (20, 30), (80, 70), 255, -1)

img_L = np.zeros((100, 100), dtype=np.uint8)
cv2.rectangle(img_L, (20, 20), (40, 80), 255, -1)
cv2.rectangle(img_L, (20, 60), (70, 80), 255, -1)

img_hole = np.zeros((100, 100), dtype=np.uint8)
cv2.rectangle(img_hole, (20, 20), (80, 80), 255, -1)
cv2.rectangle(img_hole, (35, 35), (65, 65), 0, -1)

img_circle = np.zeros((100, 100), dtype=np.uint8)
cv2.circle(img_circle, (50, 50), 30, 255, -1)

img_ring = np.zeros((100, 100), dtype=np.uint8)
cv2.circle(img_ring, (50, 50), 35, 255, -1)
cv2.circle(img_ring, (50, 50), 20, 0, -1)

show_images_grid(
    [img_rectangle, img_L, img_hole, img_circle, img_ring],
    ['Prostokąt', 'Litera L', 'Prostokąt z dziurą', 'Koło', 'Pierścień'],
    rows=2, cols=3
)

print("✓ Obrazy testowe utworzone")

## 3. Zadania do wykonania <a id='zadania'></a>

### Zadanie 1: Transformata Hit-or-Miss

#### Zadanie 1.1: Implementacja funkcji `hit_or_miss`

Zaimplementuj transformatę Hit-or-Miss dla pary elementów strukturalnych 2-wartościowych.

**Algorytm:**
1. Wykonaj erozję obrazu elementem `kernel_hit`
2. Wykonaj erozję dopełnienia obrazu elementem `kernel_miss`
3. Wynik = iloczyn logiczny (AND) obu erozji


In [ ]:
def hit_or_miss(image, kernel_hit, kernel_miss):
    """
    Transformata Hit-or-Miss.
    
    Args:
        image: obraz binarny (0/255)
        kernel_hit: SE dla pierwszego planu (piksele = 1)
        kernel_miss: SE dla tła (piksele = 1)
    
    Returns:
        Obraz z wykrytymi wzorcami
    """
    # TODO: uzupełnij implementację
    pass


#### Test Zadania 1.1


In [ ]:
test_img = np.zeros((7, 7), dtype=np.uint8)
test_img[1:3, 1:4] = 255

se_hit = np.array([[0, 0, 0], [0, 1, 1], [0, 1, 0]], dtype=np.uint8)
se_miss = np.array([[1, 1, 1], [1, 0, 0], [1, 0, 0]], dtype=np.uint8)

result = hit_or_miss(test_img, se_hit, se_miss)

if result is not None:
    show_images([test_img, result], ['Obraz testowy', 'Wykryty narożnik'])
    print(f"Liczba wykrytych pikseli: {np.sum(result > 0)}")
    print("Test zaliczony!" if np.sum(result > 0) > 0 else "Sprawdź implementację")


#### Zadanie 1.2: Wykrywanie narożników

Zaimplementuj funkcję wykrywającą wszystkie 4 typy narożników w obrazie.

**Wskazówka:** Przygotuj 4 pary elementów strukturalnych (hit, miss) dla każdego typu narożnika.


In [ ]:
def detect_corners(image):
    """
    Wykrywa wszystkie 4 typy narożników.
    
    Returns:
        dict: słownik z wykrytymi narożnikami
    """
    corners = {}
    # TODO: uzupełnij implementację
    return corners


#### Test Zadania 1.2


In [ ]:
corners = detect_corners(img_rectangle)

if corners:
    images = [img_rectangle] + list(corners.values())
    titles = ['Oryginał', 'Lewy górny', 'Prawy górny', 'Lewy dolny', 'Prawy dolny']
    show_images(images, titles, figsize=(20, 4))
    total = sum(np.sum(c > 0) for c in corners.values())
    print(f"Łączna liczba wykrytych narożników: {total}")
    print("Test zaliczony!" if total == 4 else "Sprawdź implementację")


---

### Zadanie 2: Ekstrakcja konturów

#### Zadanie 2.1: Kontur zewnętrzny

**Wzór:** $\beta_e(A) = A - (A \ominus B)$


In [ ]:
def boundary_external(image, kernel):
    # TODO: uzupełnij implementację
    pass


#### Zadanie 2.2: Kontur wewnętrzny

**Wzór:** $\beta_i(A) = (A \oplus B) - A$


In [ ]:
def boundary_internal(image, kernel):
    # TODO: uzupełnij implementację
    pass


#### Zadanie 2.3: Gradient morfologiczny

**Wzór:** $\beta(A) = (A \oplus B) - (A \ominus B)$


In [ ]:
def gradient_morphological(image, kernel):
    # TODO: uzupełnij implementację
    pass


#### Test Zadania 2


In [ ]:
ext = boundary_external(img_circle, kernel_rect_3)
int_b = boundary_internal(img_circle, kernel_rect_3)
grad = gradient_morphological(img_circle, kernel_rect_3)

if ext is not None and int_b is not None and grad is not None:
    show_images([img_circle, ext, int_b, grad],
                ['Oryginał', 'Kontur zewnętrzny', 'Kontur wewnętrzny', 'Gradient'],
                figsize=(20, 5))
    print(f"Piksele konturu zewnętrznego: {np.sum(ext > 0)}")
    print(f"Piksele konturu wewnętrznego: {np.sum(int_b > 0)}")
    print(f"Piksele gradientu: {np.sum(grad > 0)}")


---

### Zadanie 3: Wypełnianie dziur

#### Zadanie 3.1: Implementacja algorytmu wypełniania dziur

**Algorytm:**
```
1. X_0 = obraz z jednym pikselem w punkcie seed_point
2. complement = negacja obrazu (tło)
3. Dopóki X_k != X_(k-1):
   a) X_k = dilate(X_(k-1), kernel)
   b) X_k = X_k AND complement
4. Wynik = image OR X_k
```


In [ ]:
def fill_holes(image, seed_point, kernel=None, max_iter=100):
    """
    Wypełnia dziurę w obrazie binarnym.
    
    Args:
        image: obraz binarny z dziurą
        seed_point: punkt startowy (y, x) wewnątrz dziury
        kernel: element strukturalny
        max_iter: maksymalna liczba iteracji
    """
    if kernel is None:
        kernel = kernel_cross_3
    # TODO: uzupełnij implementację
    pass


#### Test Zadania 3.1


In [ ]:
seed = (50, 50)
filled = fill_holes(img_hole, seed)

if filled is not None:
    show_images([img_hole, filled], ['Obraz z dziurą', 'Po wypełnieniu'])
    diff = np.sum(filled > 0) - np.sum(img_hole > 0)
    print(f"Wypełniono {diff} pikseli")
    print("Test zaliczony!" if diff > 0 else "Sprawdź implementację")


## 4. Przykłady użycia <a id='przyklady'></a>

### Przykład 1: Analiza kształtów


In [ ]:
# Wykrywanie narożników w różnych kształtach
shapes = [img_rectangle, img_L, img_circle]
names = ['Prostokąt', 'Litera L', 'Koło']

for shape, name in zip(shapes, names):
    corners = detect_corners(shape)
    if corners:
        total = sum(np.sum(c > 0) for c in corners.values())
        print(f"{name}: {total} narożników")


### Przykład 2: Ekstrakcja konturów


In [ ]:
# Porównanie metod ekstrakcji konturów
ext = boundary_external(img_L, kernel_rect_3)
int_b = boundary_internal(img_L, kernel_rect_3)
grad = gradient_morphological(img_L, kernel_rect_3)

if ext is not None:
    show_images([img_L, ext, int_b, grad],
                ['Oryginał', 'Zewnętrzny', 'Wewnętrzny', 'Gradient'],
                figsize=(20, 5))


### Przykład 3: Wypełnianie dziur


In [ ]:
# Wypełnianie dziury w pierścieniu
filled_ring = fill_holes(img_ring, (50, 50))

if filled_ring is not None:
    show_images([img_ring, filled_ring], ['Pierścień', 'Wypełniony'])


## 5. Podsumowanie <a id='podsumowanie'></a>

### Poznane operacje:

1. **Transformata Hit-or-Miss**
   - Wykrywanie wzorców w obrazach binarnych
   - Używa pary elementów strukturalnych (hit, miss)
   - Sprawdza jednocześnie obiekt i tło

2. **Wykrywanie narożników**
   - Identyfikacja punktów charakterystycznych
   - 4 typy narożników dla prostokątów
   - Przydatne w analizie kształtów

3. **Ekstrakcja konturów**
   - Kontur zewnętrzny: A - (A ⊖ B)
   - Kontur wewnętrzny: (A ⊕ B) - A
   - Gradient morfologiczny: (A ⊕ B) - (A ⊖ B)

4. **Wypełnianie dziur**
   - Algorytm iteracyjny
   - Wymaga punktu startowego
   - Przydatne w poprawie segmentacji

### Zastosowania:
- Rozpoznawanie wzorców
- Analiza kształtu obiektów
- Inspekcja wizualna
- Segmentacja obrazu
- Ekstrakcja cech geometrycznych

---

**Koniec laboratorium 8**
